In [41]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import random
import math
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import gc

In [42]:
books_file_path = "books.csv"

df_books = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "zygmunt/goodbooks-10k",
  books_file_path,
)

Using Colab cache for faster access to the 'goodbooks-10k' dataset.


In [43]:
ratings_file_path = "ratings.csv"

df_ratings = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "zygmunt/goodbooks-10k",
  ratings_file_path,
)

Using Colab cache for faster access to the 'goodbooks-10k' dataset.


In [44]:
df_full = pd.merge(df_ratings, df_books, on='book_id')

In [45]:
df_full = df_full.drop(columns=['image_url', 'small_image_url','ratings_1','ratings_2','ratings_3','ratings_4','ratings_5', 'isbn', 'isbn13', 'work_id', 'best_book_id', 'original_publication_year', 'books_count', 'title', 'language_code', 'work_ratings_count', 'work_text_reviews_count'], errors='ignore')

In [46]:
m = df_full['ratings_count'].quantile(0.90)
C = df_full['average_rating'].mean()

def weighted_rating(row, m=m, C=C):
    v = row['ratings_count']
    R = row['average_rating']
    return round((v / (v + m)) * R + (m / (v + m)) * C, 2)

df_full['score'] = df_full.apply(weighted_rating, axis=1)

In [47]:
df_full = df_full.drop(columns=['average_rating', 'ratings_count'])

In [48]:
df = df_full.sort_values('score', ascending=False)

In [49]:
display(df.head())

,book_id,user_id,rating,id,authors,original_title,score
43,1,23576,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
35,1,18361,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
20,1,10610,5,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
19,1,10335,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45
18,1,10246,4,27,"J.K. Rowling, Mary GrandPré",Harry Potter and the Half-Blood Prince,4.45


In [50]:
df.shape

(79701, 7)

In [51]:
df_books['combined_features'] = df_books['original_title'].fillna('') + ' ' + df_books['authors'].fillna('')

In [52]:
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(df_books['combined_features'])

cosine_sim_item_based = cosine_similarity(tfidf_matrix)
indices_item_based = df_books.reset_index().drop_duplicates(subset='original_title', keep='first').set_index('original_title')['index']

In [53]:
def get_item_based_recommendations(title, num_recommendations=5):
    idx = indices_item_based[title]

    sim_scores = list(enumerate(cosine_sim_item_based[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:num_recommendations+1]

    book_indices = [i[0] for i in sim_scores]

    return df['original_title'].iloc[book_indices]

In [54]:
print("Рекомендации:")
display(get_item_based_recommendations('The Hobbit or There and Back Again'))

Рекомендации:


,original_title
15249,Pride and Prejudice
39295,The Pillars of the Earth
128,Harry Potter and the Order of the Phoenix
10975,La sombra del viento
1754,The Lord of the Rings


In [55]:
movie_titles = dict(zip(df_books['book_id'], df_books['original_title']))

In [56]:
user_ids = df_ratings['user_id'].unique()
book_ids = df_ratings['book_id'].unique()

user_to_idx = {uid: i for i, uid in enumerate(user_ids)}
book_to_idx = {bid: i for i, bid in enumerate(book_ids)}

rows = df_ratings['user_id'].map(user_to_idx).values
cols = df_ratings['book_id'].map(book_to_idx).values
data = df_ratings['rating'].values.astype(np.float32)

In [57]:
user_movie_matrix = csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(book_ids)), dtype=np.float32)

In [58]:
def get_user_based_recommendations(user_id, num_recommendations=5):
    if user_id not in user_ids:
        popular = df_ratings.groupby('book_id')['score'].mean().sort_values(ascending=False).head(num_recommendations)
        return pd.Series([movie_titles.get(mid, f'Книга {mid}') for mid in popular.index])

    user_idx = np.where(user_ids == user_id)[0][0]

    user_vector = user_movie_matrix[user_idx]
    similarities = cosine_similarity(user_vector, user_movie_matrix).flatten()

    similar_indices = np.argsort(similarities)[-21:-1][::-1]
    similar_users = user_ids[similar_indices]
    similar_scores = similarities[similar_indices]

    watched = set(df_ratings[df_ratings['user_id'] == user_id]['book_id'].values)

    scores = {}

    ratings_by_user = df_ratings.groupby('user_id')

    for uid, sim in zip(similar_users, similar_scores):
        user_ratings = ratings_by_user.get_group(uid)

        for book_id, rating in zip(user_ratings['book_id'], user_ratings['rating']):
            if book_id not in watched:
                if book_id not in scores:
                    scores[book_id] = []
                scores[book_id].append(rating * sim)

    avg_scores = {book_id: np.mean(scores_list) for book_id, scores_list in scores.items()}

    top = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)[:num_recommendations]

    del similarities, similar_indices, similar_scores
    gc.collect()

    return pd.Series([movie_titles.get(book_id, f'Книга {book_id}') for book_id, _ in top])

In [59]:
print("Рекомендации для пользователя 1:")
print(get_user_based_recommendations(1, 5))

print("\nРекомендации для нового пользователя:")
print(get_user_based_recommendations(9999, 5))

Рекомендации для пользователя 1:
0    Книга 7457
1    Книга 5325
2    Книга 9459
3    Книга 7907
4            V.
dtype: object

Рекомендации для нового пользователя:
0    Книга 4088
1    Книга 8368
2    Книга 8522
3    Книга 7369
4    Книга 9483
dtype: object
